In [1]:
import pandas as pd

df = pd.read_csv("data.csv")

# default column
df["mice"] = False

In [2]:
df

,Sanjay Sir Markings,Sangeetha Comments,Category,Unnamed: 3,Name Of The Company,Contact Name,Designation,Address 1,Address 2,City,...,Email 3,Mobile 3,Unnamed: 20,Unnamed: 21,Email 1.1,domain,domain2,Unnamed: 25,Unnamed: 26,mice
0,NaN,NaN,NaN,NaN,1497 Originals senses,Mr Daxesh Badheka,co-Founder,"A-209 Mondeal Heights Near Wide Angle, Next to...",NaN,Ahmedabad,...,NaN,NaN,NaN,NaN,daxesh,osenses.in,NaN,NaN,NaN,False
1,NaN,NaN,NaN,Yes,360 Degree Travel Concierge LLP,Ms Julie Lewis/Munifa Sareshwala,Partner,"Besides Binori Hotel, Binori Ambit, 203, Sarkh...",NaN,Ahmedabad,...,NaN,NaN,NaN,NaN,enquiry,360travelcon.com,NaN,NaN,NaN,False
2,NaN,NaN,NaN,NaN,7 Planet Holidays,Mr Dhrupesh Patel,Managing Director,"Ls no 122 Nr, shivalik villa, angan cross road","Rajpath rangoli road, ambli",Ahmedabad,...,NaN,NaN,NaN,NaN,dhrupesh,7planetholidays.com,NaN,NaN,NaN,False
3,NaN,NaN,NaN,NaN,7 seas vacations,Mr R.B Patel,Proprietor,"401 Pushpam Mall, 100 Feet",Anandnagar Road,Ahmedabad,...,NaN,NaN,NaN,NaN,7seasvacationsindia,gmail.com,NaN,NaN,NaN,False
4,NaN,NaN,NaN,NaN,99 Destinations,Ms Manisha,Senior Operations,"Golden Arc, G2/G3, Atabhai Rd, Bhavnagar, Gujarat",NaN,Ahmedabad,...,NaN,NaN,NaN,NaN,trips,99destinations.in,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,1.0,NaN,Gujarat Chapter- TAFI-Sec,NaN,Bleach Tourism Pvt. Ltd.,Mr. Bijal J Shah,Secretary,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,bijal,bleachtourism.com,NaN,NaN,NaN,False
673,1.0,NaN,Gujarat Chapter- TAFI-Treasurer,NaN,Sanskar Travels,Mr. Rajneesh Bhatnagar,Treasurer,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,sanskartravel,ymail.com,NaN,NaN,NaN,False
674,NaN,NaN,Gujarat Chapter- TAFI,NaN,Zoom Travelling Pvt. Ltd.,Mr. Himanshu Patel,Committee Member,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,ops,zoomtravelling.co.in,NaN,NaN,NaN,False
675,NaN,NaN,Gujarat Chapter- TAFI,NaN,Royal Tours & Travels,Mr. Keyur Gondalia,Committee Member,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,royaltoursntravel,gmail.com,NaN,NaN,NaN,False


In [1]:
import pandas as pd
import requests
import urllib.parse
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

df = pd.read_csv("data.csv")
df["mice"] = False

HEADERS = {"User-Agent": "Mozilla/5.0"}

EMAIL_PROVIDERS = ["gmail.com", "yahoo.com", "ymail.com", "hotmail.com", "outlook.com"]


def is_email_provider(domain):
    if pd.isna(domain):
        return False
    domain = str(domain).lower()
    return any(p in domain for p in EMAIL_PROVIDERS)


# -----------------------------
# GET ALL INTERNAL LINKS
# -----------------------------
def get_internal_links(base_url, soup):
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full_url = urljoin(base_url, href)

        # keep only same domain
        if urlparse(full_url).netloc == urlparse(base_url).netloc:
            links.add(full_url)

    return list(links)


# -----------------------------
# CHECK PAGE FOR MICE
# -----------------------------
def page_has_mice(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=8)
        if r.status_code != 200:
            return False

        soup = BeautifulSoup(r.text, "html.parser")
        text = soup.get_text(separator=" ").lower()

        return "mice" in text, soup

    except:
        return False, None


# -----------------------------
# CRAWLER (MULTI PAGE)
# -----------------------------
def crawl_domain_for_mice(domain):
    try:
        if not domain.startswith("http"):
            base_url = "https://" + domain
        else:
            base_url = domain

        print(f"    🌐 Opening: {base_url}")

        r = requests.get(base_url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return False

        soup = BeautifulSoup(r.text, "html.parser")

        # check homepage
        text = soup.get_text(separator=" ").lower()
        if "mice" in text:
            return True

        # collect internal links
        links = get_internal_links(base_url, soup)
        links = links[:10]  # limit crawl

        print(f"    🔗 Found {len(links)} internal links")

        for idx, link in enumerate(links):
            print(f"      → Crawling [{idx+1}/{len(links)}]: {link}")

            has_mice, _ = page_has_mice(link)
            if has_mice:
                print("      ✅ MICE FOUND on page!")
                return True

        return False

    except Exception as e:
        print(f"    ❌ Error: {e}")
        return False


# -----------------------------
# SEARCH FOR EMAIL DOMAINS
# -----------------------------
def search_and_check_mice(query):
    try:
        url = "https://html.duckduckgo.com/html/?q=" + urllib.parse.quote(query)

        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        text = soup.get_text(separator=" ").lower()
        return "mice" in text

    except:
        return False


# -----------------------------
# MAIN LOOP
# -----------------------------
total = len(df)

for i, row in df.iterrows():
    company = row.get("Name Of The Company")
    domain = row.get("domain")

    print(f"\n[STEP {i+1}/{total}] {company}")

    found = False

    if is_email_provider(domain):
        print("  📧 Email domain → search mode")

        query = f"{company} MICE meetings incentives conferences exhibitions"
        print(f"  🔍 Query: {query}")

        found = search_and_check_mice(query)

    else:
        print("  🌍 Website domain → crawler mode")

        found = crawl_domain_for_mice(domain)

    df.at[i, "mice"] = found

    print(f"  ✔ RESULT: {found}")


df.to_csv("data_with_mice.csv", index=False)
print("\nDONE ✔ Saved file")


[STEP 1/677] 1497 Originals senses
  🌍 Website domain → crawler mode
    🌐 Opening: https://osenses.in
  ✔ RESULT: False

[STEP 2/677] 360 Degree Travel Concierge LLP
  🌍 Website domain → crawler mode
    🌐 Opening: https://360travelcon.com
    🔗 Found 10 internal links
      → Crawling [1/10]: https://360travelcon.com/@360.html
      → Crawling [2/10]: https://360travelcon.com/australia.html
      → Crawling [3/10]: https://360travelcon.com/republic-of-singapore.html
      → Crawling [4/10]: https://360travelcon.com/destinations.html
      → Crawling [5/10]: https://360travelcon.com/north-america.html
      → Crawling [6/10]: https://360travelcon.com/west-coast-usa.html
      → Crawling [7/10]: https://360travelcon.com/corporate.html
      ✅ MICE FOUND on page!
  ✔ RESULT: True

[STEP 3/677] 7 Planet Holidays
  🌍 Website domain → crawler mode
    🌐 Opening: https://7planetholidays.com
    🔗 Found 2 internal links
      → Crawling [1/2]: https://7planetholidays.com#
      → Crawling [